# Limpieza de `Injury` y `Species` - Shark Attack (GSAF)

Funciones de limpieza para las columnas **Injury** y **Species**. Cada función solo toca su columna y devuelve la tabla.

In [11]:
import numpy as np
import pandas as pd

## Cargar el dataset

Cambia la ruta por la de tu archivo.

In [12]:
df = pd.read_excel("GSAF5.xls")
df.shape

(7125, 23)

## Funciones de limpieza

In [13]:
def limpiar_columna(serie):
    # Pasa la columna a tipo texto de pandas, que admite nulos sin dar errores
    s = serie.astype("string")

    # Borra todo lo que esté entre paréntesis, incluidos los paréntesis: "Bull shark (2m)" -> "Bull shark "
    s = s.str.replace(r"\(.*?\)", "", regex=True)

    # Borra los signos de interrogación y las comillas: '"Tiger shark?"' -> "Tiger shark"
    s = s.str.replace(r'[?"“”]', "", regex=True)

    # Cambia cualquier grupo de espacios o tabuladores seguidos por un solo espacio: "Tiger   shark" -> "Tiger shark"
    s = s.str.replace(r"\s+", " ", regex=True)

    # Quita espacios y signos (. ; ,) sobrantes al principio y al final: " Tiger shark. " -> "Tiger shark"
    s = s.str.strip(" .;,")

    # Primera letra en mayúscula y el resto en minúscula: "TIGER SHARK" -> "Tiger shark"
    s = s.str.capitalize()

    # Los textos que no aportan información (o vacíos) se unifican como "Unknown"
    # (?i) = ignora mayúsculas/minúsculas, necesario porque ya hemos aplicado capitalize()
    s = s.replace(r"(?i)^(unknown|not stated|not specified|undetermined|unidentified|no details|nan|)$",
                  "Unknown", regex=True)

    # Los nulos que quedan (NaN) también pasan a "Unknown"
    return s.fillna("Unknown")


def limpiar_injury(df):
    df = df.copy()
    df["Injury"] = limpiar_columna(df["Injury"])
    return df


def limpiar_species(df):
    df = df.copy()
    columna = "Species " if "Species " in df.columns else "Species"
    s = limpiar_columna(df[columna])

    # Registros que no indican una especie: dudosos, no confirmados o inválidos -> "Unknown"
    s = s.replace(r"(?i).*(involvement|involment|invovlement|questionable|invalid|unconfirmed|not confirmed).*",
                  "Unknown", regex=True)

    # Solo tamaño sin especie ("6' shark", "1.8 m [6'] shark", "A small shark") -> "Unknown"
    s = s.replace(r"(?i)^(?:[\d.\s\[\]'/-]|\b(?:m|ft|feet|to|a|an|the|small|large|big|juvenile)\b)*sharks?$",
                  "Unknown", regex=True)

    df[columna] = s
    return df

## Aplicar la limpieza

In [14]:
df_original = df.copy()   # copia para comparar antes/después

df = limpiar_injury(df)
df = limpiar_species(df)

## Comprobaciones

In [26]:
import pandas as pd

col = df["Injury"].astype("string")
serie = col.dropna()

print("=== INJURY: COMPROBACIONES ===")
print("Filas totales:", len(col))

# 1) Nulos reales y nulos disfrazados
marcadores = ["", "?", "unknown", "unk", "n/a", "na", "none", "null", "not known", "not reported"]
print("\n1) Nulos")
print("  Nulos reales:", col.isna().sum(), f"({col.isna().mean() * 100:.1f}%)")
print("  Nulos disfrazados:", serie.str.strip().str.lower().isin(marcadores).sum())

# 2) Espacios y formato
print("\n2) Formato")
print("  Con espacios al inicio/final:", serie.ne(serie.str.strip()).sum())
print("  Con espacios dobles:", serie.str.contains(r"\s{2,}", regex=True).sum())
print("  Con saltos de línea o tabulaciones:", serie.str.contains(r"[\n\r\t]", regex=True).sum())
print("  Con '?':", serie.str.contains(r"\?", regex=True).sum())
print("  Con paréntesis:", serie.str.contains(r"[()]", regex=True).sum())

# 3) Variantes por mayúsculas/minúsculas
print("\n3) Variantes")
print("  Únicos:", serie.nunique())
print("  Únicos normalizados (strip + lower):", serie.str.strip().str.lower().nunique())

# 4) Longitud
longitud = serie.str.len()
print("\n4) Longitud")
print(longitud.describe().round(1).to_string())
print("  Muy cortos (<=3 caracteres):")
print(serie[longitud <= 3].value_counts().head(10).to_string())
print("  Los 5 más largos:")
print(serie[longitud.nlargest(5).index].to_string())

# 5) Valores más frecuentes
print("\n5) Top 20 valores")
print(serie.value_counts().head(20).to_string())

# 6) Valores que aparecen una sola vez (candidatos a variantes o errores)
conteos = serie.value_counts()
print("\n6) Valores que aparecen una sola vez:", (conteos == 1).sum(), "de", len(conteos))

# 7) Caracteres raros (fuera de letras, números y puntuación habitual)
raros = serie[serie.str.contains(r"[^\w\s.,;:'\"/&%+\-]", regex=True)]
print("\n7) Con caracteres raros:", len(raros))
print(raros.head(10).to_string())

# 8) Palabras clave útiles para análisis
print("\n8) Palabras clave")
for palabra in ["fatal", "no injury", "minor", "serious", "severe", "survived", "unconfirmed"]:
    print(f"  {palabra!r}: {serie.str.contains(palabra, case=False, regex=False).sum()}")

=== INJURY: COMPROBACIONES ===
Filas totales: 7125

1) Nulos
  Nulos reales: 0 (0.0%)
  Nulos disfrazados: 86

2) Formato
  Con espacios al inicio/final: 0
  Con espacios dobles: 0
  Con saltos de línea o tabulaciones: 0
  Con '?': 0
  Con paréntesis: 0

3) Variantes
  Únicos: 4083
  Únicos normalizados (strip + lower): 4083

4) Longitud
count    7125.0
mean       30.8
std        24.8
min         5.0
25%        13.0
50%        24.0
75%        41.0
max       232.0
  Muy cortos (<=3 caracteres):
Series([], )
  Los 5 más largos:
6224    Fatal, 101 people perished, including commande...
5325    U.s. destroyer john w. weeks, 6 miles offshore...
6910    Left arm severed 2.5 from elbow, groin, abdome...
3954    His feet were bitten by sharks, but he was res...
5292    His remains were recovered from a huge shark c...

5) Top 20 valores
Injury
Fatal                        883
Foot bitten                  100
Survived                      98
No injury                     96
Unknown             